In [50]:
from re import L
import h5py
import torch
import glob
import sys
import os 
import pickle
import numpy as np
from pathlib import Path

In [51]:
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 

In [52]:
word_list = list(class_map.keys())

In [53]:
import nltk
from nltk.corpus import wordnet

In [54]:
# Download WordNet if not already downloaded
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

def remove_tenses(words):
    # Initialize a lemmatizer
    lemmatizer = nltk.stem.WordNetLemmatizer()

    # Define a function to get the part of speech for each word
    def get_wordnet_pos(word):
        tag = nltk.pos_tag([word])[0][1][0].upper()
        tag_dict = {"J": wordnet.ADJ, 
                    "N": wordnet.NOUN,
                    "V": wordnet.VERB, 
                    "R": wordnet.ADV}
        return tag_dict.get(tag, wordnet.NOUN)  # Default to noun if not found

    # Lemmatize each word based on its part of speech and store in a set to remove duplicates
    base_words = set()
    for word in words:
        base_word = lemmatizer.lemmatize(word)#, pos=get_wordnet_pos(word))
        base_words.add(base_word)

    return list(base_words)

[nltk_data] Downloading package wordnet to /home/imgriff/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/imgriff/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [55]:
word_list = [word for word in word_list if "'" not in word]
len(word_list)

784

In [56]:
base_words = remove_tenses(word_list)
len(base_words)

742

In [57]:
shorter_words = [word for word in base_words if 4 <= len(word) <= 7]

In [58]:
len(shorter_words)

551

In [59]:
shorter_words.remove('centre')

In [60]:
shorter_words.sort()
print(shorter_words)

['about', 'above', 'access', 'across', 'action', 'active', 'added', 'africa', 'after', 'again', 'against', 'airport', 'album', 'allowed', 'almost', 'alone', 'along', 'already', 'always', 'america', 'among', 'ancient', 'animal', 'another', 'appear', 'appears', 'area', 'around', 'artist', 'asked', 'attack', 'author', 'award', 'based', 'battle', 'beach', 'became', 'because', 'become', 'before', 'began', 'behind', 'being', 'believe', 'below', 'better', 'between', 'bird', 'black', 'blood', 'board', 'book', 'border', 'bridge', 'bright', 'bring', 'british', 'brother', 'brought', 'brown', 'built', 'buried', 'called', 'canada', 'cannot', 'capital', 'career', 'carried', 'case', 'cause', 'caused', 'center', 'central', 'certain', 'change', 'changed', 'charge', 'charles', 'chief', 'child', 'china', 'chinese', 'church', 'city', 'civil', 'class', 'clear', 'close', 'closed', 'coach', 'coast', 'college', 'color', 'come', 'coming', 'common', 'company', 'complex', 'concept', 'control', 'could', 'council'

In [61]:
# make sure shorter_words are actually in word_list
short_word_set = set(shorter_words)
word_list_set = set(word_list)
included_words = short_word_set.intersection(word_list_set)
print(len(included_words))


# find words in shorter_words that are not in word_list
missing_words = short_word_set - word_list_set
print(len(missing_words.intersection(short_word_set)))

## Make sure included words are actually in the word_list
print(len(included_words.intersection(word_list_set)))
for word in included_words:
    assert word in word_list, f"{word} not in word_list"

505
45
505


In [62]:
included_words_list = list(included_words)
for word in included_words:
    # if word == 'centre' or word == 'center':
    #     print(word)
    for prefix in ['s', 'ed', 'er', 'd']:
        if word + prefix in included_words:
            print(word)
            print(word + prefix)
            included_words_list.remove(word + prefix)

cause
caused
change
changed
appear
appears
offer
offered
young
younger
other
others
force
forced
place
placed
small
smaller
close
closed
start
started
cover
covered
found
founded


In [63]:
len(included_words_list)

492

In [64]:
included_words_list.sort()
print(included_words_list)

['about', 'above', 'access', 'across', 'action', 'active', 'added', 'africa', 'after', 'again', 'against', 'airport', 'album', 'allowed', 'almost', 'alone', 'along', 'already', 'always', 'america', 'among', 'ancient', 'another', 'appear', 'around', 'artist', 'asked', 'attack', 'author', 'award', 'based', 'battle', 'beach', 'became', 'because', 'become', 'before', 'began', 'behind', 'being', 'believe', 'below', 'better', 'between', 'black', 'blood', 'board', 'border', 'bridge', 'bright', 'bring', 'british', 'brother', 'brought', 'brown', 'built', 'buried', 'called', 'canada', 'cannot', 'capital', 'career', 'carried', 'cause', 'center', 'central', 'certain', 'change', 'charge', 'charles', 'chief', 'child', 'china', 'chinese', 'church', 'civil', 'class', 'clear', 'close', 'coach', 'coast', 'college', 'color', 'coming', 'common', 'company', 'complex', 'concept', 'control', 'could', 'council', 'country', 'county', 'couple', 'course', 'court', 'cover', 'created', 'creek', 'cross', 'culture',

In [65]:
## Manually remove specific words
included_words_list.remove('there')
included_words_list.remove('killed')
included_words_list.remove('death')
included_words_list.remove('blood')
# included_words_list.remove('covered')


In [66]:

len(included_words_list)

488

In [67]:
with open("word_list_for_binaural_sentences.pkl", "wb") as output:
    pickle.dump(included_words_list, output)

## Use OpenAI GPT to generate text for target words

Need two target sentences per word

Need two cue sentences use non-taret words, but have the same general length 

In [1]:
import pickle
with open("word_list_for_binaural_sentences.pkl", "rb") as f:
   target_word_list =  pickle.load(f)

In [2]:
from io import StringIO
# import typing_extensions 
from openai import OpenAI




### Generation will take rougly 1 minute

In [18]:

messages = [
    {"role":"system", "content": f"You are a writer tasked with writing short sentences and will return a python dictionary. Given a list of target words, generate two sentences that include the target word as the middle word." \
                      # "The other words in each sentence should be under 7 letters long." \
                      "For each sentence, follow these steps. Step 1: pick either 2 or 3 words to occur before the target word. Step 2: Use the sentence from Step 1 and pick either 2 or 3 more words to complete the sentence."\
                      "There must always be no more than 4 words before or 4 words after the target word."\
                      "The sentences must be complete sentences that are gramatical and make sense." \
                      "If the target word is the name of a nation, use it as the middle word." \
                      "Return the results as a dictionary with each word as a key and a python list containing both sentences as the corresponding value." },
    
    # {"role":"system", "content": f"Here is the list of words you will be working with: {target_word_list}"}
    # {"role":"system", "content": f"Do not reuse words from this list in other sentences. They should only occur in their own sentences. Make sure these words in the list appear only in the middle of their sentence."},

    
# {"role":"user", "content": "An example of a valid example given the target word 'happy' would be {'happy': ['They heard happy news today', 'They say happy thoughts work']}"},
# {"role":"user", "content": "An example of of an invalid example the target word 'happy' would be {'happy': ['Happy days.', 'Today I heard something happy.']}"},
{"role":"user", "content": f"Write two sentences for each word in this list of target words {target_word_list[:20]}."},
]

response = client.chat.completions.create(
model="gpt-4-turbo",
# response_format={ "type": "json_object" },
# model = "gpt-4",
  messages=messages,
)
  # generations.append(response.choices[0].message.content)
print(response.choices[0].message.content)


{
    "about": [
        "She learned about him yesterday.",
        "They forgot about the plan sooner."
    ],
    "above": [
        "The eagle soared above the clouds.",
        "Stars twinkled above us brightly."
    ],
    "access": [
        "Many people need access to resources.",
        "He gained access to the files yesterday."
    ],
    "across": [
        "She walked across the street confidently.",
        "He looked across the valley quietly."
    ],
    "action": [
        "The movie's action drew crowds immediately.",
        "Your swift action saved lives earlier."
    ],
    "active": [
        "She remained active throughout the years.",
        "The volcano is active again now."
    ],
    "added": [
        "She added sugar to coffee quickly.",
        "He just added another point calmly."
    ],
    "africa": [
        "We traveled through Africa last summer.",
        "Civilizations originated in Africa anciently."
    ],
    "after": [
        "He arrived soon

In [66]:
import json

In [67]:
with open('words_and_sentences.json', 'w') as f:
    json.dump(words_and_sentences, f)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)

